# Regression by Minimizing Absolute Deviation — Experimental Workflow

This notebook contains only the experimental workflow: data preparation, model fitting, evaluation, visualizations, controlled experiments, runtime benchmarking, validation, and generated outputs.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display, Image

from absolute_deviation.data import DATASETS, load_dataset
from absolute_deviation.experiments import (
    FIGURE_DIR,
    RESULT_DIR,
    run_original_data,
    run_contamination_experiment,
    run_error_distribution_experiment,
    run_runtime_benchmark,
    validate_lad_solver,
)
from absolute_deviation.plotting import (
    generate_all_figures,
    plot_real_dataset_predictions,
    plot_hbk_multivariate_inlier_outlier,
    plot_unlabeled_multivariate_residual_space,
)


## 1. Data preparation


In [ ]:
dataset_rows = []
for name in DATASETS:
    X, y, predictors = load_dataset(name)
    dataset_rows.append({
        "dataset": name,
        "observations": len(y),
        "predictors": X.shape[1],
        "predictor_names": ", ".join(predictors),
    })

dataset_summary = pd.DataFrame(dataset_rows)
display(dataset_summary)


## 2. Fit OLS and LAD on the empirical datasets


In [ ]:
original_metrics, original_coefficients = run_original_data()

display(original_metrics)
display(original_coefficients.head(20))


## 3. Empirical visualizations


In [ ]:
plot_real_dataset_predictions()
plot_hbk_multivariate_inlier_outlier()
plot_unlabeled_multivariate_residual_space("boston_housing", "Boston Housing")
plot_unlabeled_multivariate_residual_space("concrete_strength", "Concrete Strength")

empirical_figures = [
    "boston_housing_actual_vs_fitted.png",
    "concrete_strength_actual_vs_fitted.png",
    "hbk_actual_vs_fitted.png",
    "hbk_multivariate_inlier_outlier.png",
    "boston_housing_multivariate_residuals.png",
    "concrete_strength_multivariate_residuals.png",
]

for filename in empirical_figures:
    path = FIGURE_DIR / filename
    if path.exists():
        display(Image(filename=str(path)))


## 4. Large response-error experiment


In [ ]:
contamination_metrics, contamination_changes = run_contamination_experiment()

contamination_summary = (
    contamination_metrics
    .groupby(["contamination_fraction", "model"], as_index=False)[["SSE", "SAE", "MAE", "RMSE"]]
    .mean()
)
display(contamination_summary)

shift_summary = (
    contamination_changes
    .groupby(["contamination_fraction", "model"], as_index=False)[
        ["coefficient_shift_l2", "prediction_shift_mae"]
    ]
    .mean()
)
display(shift_summary)


## 5. Error-distribution experiment


In [ ]:
distribution_results = run_error_distribution_experiment()

distribution_summary = (
    distribution_results
    .groupby(["distribution", "model"], as_index=False)[
        ["coefficient_error_l2", "SSE", "SAE", "MAE", "RMSE"]
    ]
    .median()
)
display(distribution_summary)


## 6. Runtime benchmark


In [ ]:
runtime_results = run_runtime_benchmark()

runtime_summary = (
    runtime_results
    .groupby(["n", "p", "model"], as_index=False)["runtime_seconds"]
    .median()
)
display(runtime_summary)


## 7. Solver validation


In [ ]:
validation_results = validate_lad_solver()
display(validation_results)


## 8. Generate final figures and outputs


In [ ]:
generate_all_figures()

result_files = sorted(path.name for path in RESULT_DIR.glob("*"))
figure_files = sorted(path.name for path in FIGURE_DIR.glob("*"))

display(pd.DataFrame({"result_files": pd.Series(result_files)}))
display(pd.DataFrame({"figure_files": pd.Series(figure_files)}))


## 9. Experimental summary


In [ ]:
summary = original_metrics.pivot(
    index="dataset",
    columns="model",
    values=["SSE", "SAE", "MAE", "RMSE"],
)
display(summary)


The experiment compares OLS and LAD on the same empirical data, then examines their behavior under deliberately introduced large response errors, different error distributions, and increasing computational workload. The validation step checks that each implementation minimizes its intended error criterion.
